# Computational RF: Signal Basics
Recreated from our exercises. Run cells in order using Shift + Enter.
Synthetic signals; not antenna measurements.
Requirements: numpy, pandas, scipy, matplotlib.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks


## 1. A 1 kHz sine wave
20,000 samples per second. Display the first 10 ms.


In [ ]:
fs = 20_000
n = 2000
t = np.arange(n) / fs
x = np.sin(2 * np.pi * 1000 * t)
plt.figure(figsize=(9, 4))
plt.plot(t[:200] * 1000, x[:200])
plt.xlabel("Time (ms)")
plt.ylabel("Amplitude")
plt.title("1 kHz sine wave")
plt.grid(True)
plt.tight_layout()
plt.show()


## 2. One-sided amplitude spectrum
Exclude DC and Nyquist from doubling for even-length signals.


In [ ]:
def amplitude_spectrum(x, fs):
    n = len(x)
    frequencies = np.fft.rfftfreq(n, d=1 / fs)
    amplitude = np.abs(np.fft.rfft(x)) / n
    if n % 2 == 0:
        amplitude[1:-1] *= 2
    else:
        amplitude[1:] *= 2
    return frequencies, amplitude

frequencies, amplitude = amplitude_spectrum(x, fs)
plt.figure(figsize=(9, 4))
plt.plot(frequencies, amplitude)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("Single-tone spectrum")
plt.xlim(0, 5000)
plt.grid(True)
plt.show()
print("Peak frequency (Hz):", frequencies[np.argmax(amplitude)])


## 3. Two tones and CSV export
Expected peaks: 1,000 Hz at amplitude 1.0 and 3,000 Hz at amplitude 0.5. Both frequencies fall exactly on FFT bins.


In [ ]:
x = np.sin(2 * np.pi * 1000 * t) + 0.5 * np.sin(2 * np.pi * 3000 * t)
frequencies, amplitude = amplitude_spectrum(x, fs)
results = pd.DataFrame({"frequency_Hz": frequencies, "amplitude": amplitude})
results.to_csv("two_tone_spectrum.csv", index=False)
display(results.nlargest(2, "amplitude"))


## 4. Read CSV and detect peaks
SciPy identifies local peaks with amplitude at least 0.1.


In [ ]:
saved_data = pd.read_csv("two_tone_spectrum.csv")
frequencies = saved_data["frequency_Hz"].to_numpy()
amplitude = saved_data["amplitude"].to_numpy()
peaks, properties = find_peaks(amplitude, height=0.1)
peak_results = pd.DataFrame({"frequency_Hz": frequencies[peaks], "amplitude": properties["peak_heights"]})
peak_results.to_csv("detected_peaks.csv", index=False)
display(peak_results)


## 5. Plot and save peak detection


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(frequencies, amplitude, label="Spectrum")
plt.plot(frequencies[peaks], amplitude[peaks], "ro", label="Detected peaks")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("Automatic frequency detection")
plt.xlim(0, 5000)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("detected_peaks.png", dpi=150)
plt.show()


## Practice
Change the 3 kHz component amplitude to 0.25 and rerun sections 3–5.
The notebook generates two CSV files and a PNG graph in its working folder.
scikit-rf and antenna measurements will be covered separately.
